In [ ]:
%pip install "transformers==5.17.0" "librosa==1.0.0" \
    "demucs==4.1.0" "jiwer==4.0.0" "soundfile==0.14.0" \
    --quiet


# Module M5 — เทรนโมเดลฟังเสียงจากศูนย์ + Demucs + ASR bake-off

**โมดูลนี้รันบน Kaggle GPU** — สร้าง notebook บน [kaggle.com/code](https://www.kaggle.com/code) แล้วตั้งค่า **Settings → Accelerator: GPU T4 ×2** (ใช้ตัวเดียวพอ) และ **Internet: On** ก่อนกด Run All

- เซลล์แรกติดตั้ง dependencies — **หมายเหตุ:** notebook นี้ใช้ `transformers==5.17.0` สูงกว่า pin กลางของ repo (4.57.6) เพราะ Typhoon-ASR-0.6B ต้องการ transformers >= 5.13 (โมดูลอื่นของคอร์ส M2/M3 ยังใช้ 4.57.6 ตาม `requirements-kaggle.txt`)
- torch/torchaudio **ไม่ต้องติดตั้ง** — Kaggle ให้มาแล้ว
- ส่วน KWS (เทรนจากศูนย์) รวมดาวน์โหลด + เทรน + วัดผล ≈ **30 นาทีบน T4**; ทั้งโมดูล (รวม Demucs + ASR) ≈ 45–60 นาที
- สถานะการตรวจ: โค้ดผ่าน lint และทดสอบ code path บน CPU แล้ว — **การรันจริงบน Kaggle T4 ต้องตรวจด้วยมือ 1 รอบก่อนวันอบรม** (ดู checklist ใน README)

**ที่มาและ license ของทุกอย่างที่โมดูลนี้ใช้**

| สิ่งที่ใช้ | ที่มา | license |
|---|---|---|
| Speech Commands v2 (105,829 คลิปเสียง 1 วินาที, 35 คำ) | [Google research](https://research.google/blog/launching-the-speech-commands-dataset/) · [download.tensorflow.org](http://download.tensorflow.org/data/speech_commands_v0.02.tar.gz) · paper [arXiv:1804.03209](https://arxiv.org/abs/1804.03209) | **CC BY 4.0** |
| FLEURS-Thai (dev split, ~440 คลิป) | [HF: google/fleurs](https://huggingface.co/datasets/google/fleurs) · paper [arXiv:2205.12447](https://arxiv.org/abs/2205.12447) | **CC BY 4.0** |
| Demucs — `htdemucs` (แยกเสียง 4 stems) | [GitHub: facebookresearch/demucs](https://github.com/facebookresearch/demucs) | **MIT** |
| Whisper-small — `openai/whisper-small` (244M parameters) | [HF model card](https://huggingface.co/openai/whisper-small) | **Apache-2.0** |
| Typhoon-ASR-0.6B — `typhoon-ai/typhoon-asr-qwen-0.6b-ctx` (LoRA-merged ของ Qwen3-ASR-0.6B, เทรนบนเสียงไทย ~10,800 ชั่วโมง) | [HF model card](https://huggingface.co/typhoon-ai/typhoon-asr-qwen-0.6b-ctx) | **Apache-2.0** |
| librosa / soundfile / jiwer / transformers / torchaudio | pip | ISC / BSD-3 / Apache-2.0 / Apache-2.0 / BSD-3 |


## 0. โน้ตช่วย — คำศัพท์ที่ใช้ในโมดูลนี้ (สำหรับผู้ที่ไม่ได้เรียนคอร์ส DS)

| คำ | ความหมาย |
|---|---|
| **sampling rate** | จำนวนตัวเลขที่อัดเสียงต่อวินาที — ไฟล์เสียง คือแกนเวลา + ความดันอากาศ เท่านั้น (เสียง = ตัวเลขแล้วตั้งแต่ต้น!) โมดูลนี้ใช้ 16,000 Hz |
| **spectrogram** | เสียง → หั่นเป็นหน้าต่างสั้น ๆ → คำนวณความถี่ในแต่ละหน้าต่าง → ได้ "ภาพ" 2 มิติ (เวลา × ความถี่) |
| **log-mel spectrogram** | spectrogram ที่สเกลความถี่ตามการได้ยินของคน (mel) แล้วบีบช่วงไดนามิกด้วย log — รูปแบบมาตรฐานสำหรับ โมเดลเสียง |
| **CNN (convolutional neural network)** | โมเดลที่กรองภาพด้วยชุดฟิลเตอร์เล็ก ๆ — ใช้กับ "ภาพ" log-mel ได้เหมือนใช้กับภาพถ่าย (เทคนิคเดียวกับ M1) |
| **เทรนจากศูนย์ (from scratch)** | ค่าน้ำหนักทั้งหมดเริ่มจาก random แล้ว gradient descent ปรับทีละนิดจากข้อมูลเราเอง — ไม่ยืม สมองสำเร็จรูปอย่าง transfer learning ใน M1/M3 |
| **stem** | แทร็กย่อยของเพลงที่แยกออกมา (กลอง/เบส/เครื่องดนตรีอื่น/เสียงร้อง) — งาน source separation |
| **ASR (automatic speech recognition)** | เสียงพูด → ข้อความ |
| **WER (word error rate)** | วัดความผิดพลาดของการถอดเสียง: (แทน + ใส่เกิน + ตกหล่น) / จำนวนคำอ้างอิง — ต่ำ = แม่น |
| **CER (character error rate)** | เหมือน WER แต่นับระดับตัวอักษร — สำหรับภาษาไทย (ไม่เว้นวรรคคำ) มักสื่อความดีกว่า WER |


In [ ]:
import os  # noqa: E402

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)

import numpy as np  # noqa: E402
import torch  # noqa: E402

torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch:", torch.__version__, "| device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("ไม่มี GPU — ไปตั้งค่า Settings > Accelerator: GPU T4")


## ส่วนที่ 1 — เทรน Keyword Spotting (KWS) จากศูนย์

โจทย์: คอมพิวเตอร์ "ฟัง" แล้วบอกว่าได้ยินคำสั่งอะไร — เหมือน "Hey Siri" / "Ok Google" แต่เราจะ**สร้างโมเดลเองทั้งหมด**จากข้อมูลดิบ (ไฟล์ .wav) วงจรเดิมของคอร์ส:

**ข้อมูลดิบ (wav) → แปลงเป็นตัวเลข (log-mel spectrogram) → เทรน CNN → วัดผลด้วย test set ที่โมเดลไม่เคยเห็น**

เราใช้ Speech Commands v2 ของ Google — คลิปเสียง 1 วินาทีจากอาสาสมัครหลายพันคน อ่านคำภาษาอังกฤษ 35 คำ เราจะจำแนก **10 คำสั่ง** (down/go/left/no/off/on/right/stop/up/yes) + 2 class พิเศษ: `_unknown_` (คำอื่นที่เหลือ) และ `_silence_` (เสียงห้องเงียบ ๆ)

In [ ]:
import tarfile  # noqa: E402
import urllib.request  # noqa: E402
from pathlib import Path  # noqa: E402

SC_URL = (
    "http://download.tensorflow.org/data/"
    "speech_commands_v0.02.tar.gz"
)
SC_ROOT = Path("/kaggle/working/speech_commands_v2")
ARCHIVE = "/kaggle/working/speech_commands_v0.02.tar.gz"

if not (SC_ROOT / "testing_list.txt").exists():
    if not Path(ARCHIVE).exists():
        print("กำลังดาวน์โหลด ~2.4 GB (2-5 นาที) ...")
        urllib.request.urlretrieve(SC_URL, ARCHIVE)
    print("กำลังแตกไฟล์ (1-2 นาที) ...")
    with tarfile.open(ARCHIVE) as tar:
        tar.extractall(SC_ROOT, filter="data")
    Path(ARCHIVE).unlink()
print("dataset พร้อมที่:", SC_ROOT)
print("จำนวนไฟล์ .wav:", len(list(SC_ROOT.glob("*/*.wav"))))


จุดสำคัญของชุดข้อมูลนี้: Google แถม **รายชื่อไฟล์สำหรับ validation/testing** มาให้ด้วย (`validation_list.txt` / `testing_list.txt`) — การแบ่ง train/val/test เลยเป็นแบบ **แยกตามผู้พูด** (คนเดียวกันไม่ปรากฏในสองชุด) ป้องกันการ "ท่องจำเสียงคน" ทำให้ตัวเลข test สะท้อนของจริง

เรา cap จำนวนตัวอย่างต่อ class เพื่อให้แล็บจบใน ~30 นาที (ของจริงเทรนเต็มได้ — แก้ `MAX_PER_WORD` ได้)


In [ ]:
import random  # noqa: E402

CMD_WORDS = [
    "down", "go", "left", "no", "off",
    "on", "right", "stop", "up", "yes",
]
LABELS = CMD_WORDS + ["_unknown_", "_silence_"]
MAX_PER_WORD = 1500
UNKNOWN_N = 2000

val_set = set((SC_ROOT / "validation_list.txt").read_text().split())
test_set = set((SC_ROOT / "testing_list.txt").read_text().split())

def split_of(path):
    rel = f"{path.parent.name}/{path.name}"
    if rel in test_set:
        return "test"
    if rel in val_set:
        return "val"
    return "train"

buckets = {"train": [], "val": [], "test": []}
for path in sorted(SC_ROOT.glob("*/*.wav")):
    word = path.parent.name
    if word == "_background_noise_":
        continue
    label = word if word in CMD_WORDS else "_unknown_"
    buckets[split_of(path)].append((path, label))

rng = random.Random(SEED)
by_word = {}
for path, label in buckets["train"]:
    by_word.setdefault(label, []).append((path, label))

train_pairs = []
for word in CMD_WORDS:
    train_pairs.extend(sorted(by_word.get(word, []))[:MAX_PER_WORD])
unknown_train = sorted(by_word.get("_unknown_", []))
rng.shuffle(unknown_train)
train_pairs.extend(unknown_train[:UNKNOWN_N])
val_pairs = buckets["val"]
test_pairs = buckets["test"]

splits = [
    ("train", train_pairs),
    ("val", val_pairs),
    ("test", test_pairs),
]
for name, pairs in splits:
    counts = {}
    for _, label in pairs:
        counts[label] = counts.get(label, 0) + 1
    print(f"{name}: {len(pairs)} ตัวอย่าง")
    print("   ", counts)


## 1.1 เสียง → ตัวเลข: log-mel spectrogram

คลิปเสียง 1 วินาที = ตัวเลข 16,000 ตัว (แกนเวลา) — พอสำหรับ "หู" แต่ CNN ทำงานกับ "ภาพ" ได้ดีกว่า เราจึงแปลงเป็น **log-mel spectrogram** 64 แถบความถี่ × 63 หน้าต่างเวลา — ตรรกะเดียวกับ M4 (MFCC/mel features) แต่คราวนี้เราให้**โมเดลดูภาพทั้งใบ** ไม่หยิบค่าเฉลี่ยเป็นตาราง


In [ ]:
import matplotlib.pyplot as plt  # noqa: E402
import soundfile as sf  # noqa: E402
import torchaudio  # noqa: E402

SAMPLES = 16000

mel_fn = torchaudio.transforms.MelSpectrogram(
    sample_rate=SAMPLES, n_fft=1024, hop_length=256, n_mels=64,
).to(DEVICE)
to_db = torchaudio.transforms.AmplitudeToDB().to(DEVICE)

def load_wave(path):
    wave, sr = sf.read(str(path), dtype="float32")
    if wave.ndim > 1:
        wave = wave.mean(axis=1)
    wave = wave[:SAMPLES]
    if len(wave) < SAMPLES:
        wave = np.pad(wave, (0, SAMPLES - len(wave)))
    return wave

def wave_to_mel(wave):
    x = torch.from_numpy(wave).unsqueeze(0).to(DEVICE)
    return to_db(mel_fn(x))[0].cpu().numpy()

examples = []
for want in ["yes", "no"]:
    path, _ = next(
        (p, w) for p, w in buckets["train"] if w == want
    )
    examples.append((want, load_wave(path)))

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for ax, (word, wave) in zip(axes, examples):
    mel = wave_to_mel(wave)
    ax.imshow(mel, aspect="auto", origin="lower")
    ax.set_title(f'"{word}" — log-mel 64 x {mel.shape[1]}')
fig.suptitle("เสียง 1 วินาที -> ภาพ log-mel spectrogram")
plt.tight_layout()
plt.show()
print("รูปร่าง feature ต่อ 1 คลิป:", mel.shape, "(mel x time)")


**อ่านภาพ:** แถบสว่าง = พลังงานเสียงสูง แต่ละคำมี "รอยประทับ" ต่างกันชัดเจน — โจทย์ของ CNN คือแยกรอยประทับ 12 แบบ

ต่อไป precompute feature ของทุก split ครั้งเดียว (ใช้เวลา 1–2 นาที) เพื่อให้รอบเทรนเร็ว — `_silence_` สร้างเองจากไฟล์ เสียงห้องจริงใน `_background_noise_/` โดยสุ่มช่วง 1 วินาที และสุ่มระดับความดัง


In [ ]:
def make_silence(n, rng_local):
    """สร้างคลิป _silence_ จากเสียงห้องจริง (สุ่มช่วง+ความดัง)."""
    out = []
    for _ in range(n):
        path = noise_files[rng_local.randrange(len(noise_files))]
        full, _ = sf.read(str(path), dtype="float32")
        if full.ndim > 1:
            full = full.mean(axis=1)
        start = rng_local.randrange(0, len(full) - SAMPLES - 1)
        clip = full[start:start + SAMPLES].copy()
        out.append(clip * rng_local.uniform(0.0, 1.0))
    return out

def featurize(pairs, silence_waves):
    waves = [load_wave(path) for path, _ in pairs]
    labels = [LABELS.index(w) for _, w in pairs]
    waves += silence_waves
    labels += [LABELS.index("_silence_")] * len(silence_waves)
    feats = []
    with torch.no_grad():
        for i in range(0, len(waves), 512):
            x = torch.from_numpy(np.stack(waves[i:i + 512]))
            feats.append(to_db(mel_fn(x.to(DEVICE))).cpu())
    x_all = torch.cat(feats).unsqueeze(1)
    y_all = torch.tensor(labels)
    return x_all, y_all

noise_files = sorted(
    (SC_ROOT / "_background_noise_").glob("*.wav")
)
print("ไฟล์เสียงห้อง:", [f.name for f in noise_files])

print("เตรียม train ...")
X_train, y_train = featurize(train_pairs, make_silence(400, rng))
print("เตรียม val ...")
X_val, y_val = featurize(val_pairs, make_silence(100, rng))
print("เตรียม test ...")
X_test, y_test = featurize(test_pairs, make_silence(100, rng))
print("train:", tuple(X_train.shape), "| val:",
      tuple(X_val.shape), "| test:", tuple(X_test.shape))


## 1.2 โมเดล CNN จากศูนย์

สถาปัตยกรรมเดียวกับตระกูล MNIST ใน M1: 3 ชุด `Conv2d → BatchNorm → ReLU → MaxPool` แล้วปิดด้วย 2 ชั้น fully-connected รวม ~0.5M พารามิเตอร์ — **เริ่มจากค่า random ทั้งหมด** ไม่มี pre-trained ใด ๆ

| ชั้น | ผลลัพธ์ | ทำอะไร |
|---|---|---|
| input | 1 × 64 × 63 | log-mel spectrogram |
| conv 3×3 ×16 + pool 2 | 16 × 32 × 31 | จับลายเส้นเล็ก |
| conv 3×3 ×32 + pool 2 | 32 × 16 × 15 | จับรูปทรงคำ |
| conv 3×3 ×64 + pool 2 | 64 × 8 × 7 | จับรูปแบบใหญ่ |
| flatten + linear 128 + linear 12 | 12 | คะแนนต่อ class |


In [ ]:
class KwsNet(torch.nn.Module):
    """CNN เล็ก ๆ สำหรับ 12-class keyword spotting."""

    def __init__(self, n_classes):
        super().__init__()
        self.features = torch.nn.Sequential(
            torch.nn.Conv2d(1, 16, 3, padding=1),
            torch.nn.BatchNorm2d(16),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2),
            torch.nn.Conv2d(16, 32, 3, padding=1),
            torch.nn.BatchNorm2d(32),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2),
            torch.nn.Conv2d(32, 64, 3, padding=1),
            torch.nn.BatchNorm2d(64),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(2),
        )
        self.classifier = torch.nn.Sequential(
            torch.nn.Flatten(),
            torch.nn.Linear(64 * 8 * 7, 128),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.3),
            torch.nn.Linear(128, n_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

model = KwsNet(n_classes=len(LABELS)).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"พารามิเตอร์ทั้งหมด: {n_params:,}")
print("ทดลอง forward 1 batch:",
      tuple(model(X_train[:4].to(DEVICE)).shape))


## 1.3 เทรน — gradient descent ด้วยมือของเราเอง

รอบเทรนแบบ PyTorch พื้นฐาน (ไม่มี `Trainer` ช่วย เหมือน M3 — นี่คือ "จากศูนย์" เต็มรูปแบบ): ดึง batch → ทำนาย → วัด cross-entropy loss → backward → อัปเดตน้ำหนัก ทำซ้ำ 6 epochs

กดรัน — ใช้เวลาราว **3–5 นาทีบน T4** แต่ละ epoch จะพิมพ์ loss + accuracy (train/val) ออกมา


In [ ]:
from torch.utils.data import (  # noqa: E402
    DataLoader,
    TensorDataset,
)

BATCH = 128
EPOCHS = 6
loss_fn = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

@torch.no_grad()
def accuracy(net, X, y, batch=512):
    net.eval()
    correct = 0
    for i in range(0, len(X), batch):
        logits = net(X[i:i + batch].to(DEVICE))
        pred = logits.argmax(dim=-1).cpu()
        correct += int((pred == y[i:i + batch]).sum())
    return correct / len(X)

loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=BATCH,
    shuffle=True,
)

history = []
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = loss_fn(model(xb), yb)
        loss.backward()
        optimizer.step()
        total_loss += float(loss.item()) * len(yb)
    train_loss = total_loss / len(y_train)
    train_acc = accuracy(model, X_train, y_train)
    val_acc = accuracy(model, X_val, y_val)
    history.append((train_loss, train_acc, val_acc))
    print(
        f"epoch {epoch}/{EPOCHS} loss={train_loss:.3f} "
        f"acc train={train_acc:.3f} val={val_acc:.3f}"
    )


**อ่านตัวเลข:** loss ต้องลดลงราย epoch และ val accuracy ควรขึ้นไปแตะ ~0.90+ ถ้า val ลดช้ากว่า train มากหรือเริ่มเบนขึ้น คือสัญญาณ overfit (แก้ด้วยข้อมูลมากขึ้น / epochs น้อยลง / dropout มากขึ้น)


In [ ]:
epochs = list(range(1, EPOCHS + 1))
losses = [h[0] for h in history]
accs_tr = [h[1] for h in history]
accs_val = [h[2] for h in history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.5))
ax1.plot(epochs, losses, marker="o")
ax1.set_title("train loss (cross-entropy)")
ax1.set_xlabel("epoch")
ax2.plot(epochs, accs_tr, marker="o", label="train")
ax2.plot(epochs, accs_val, marker="s", label="val")
ax2.set_title("accuracy")
ax2.set_xlabel("epoch")
ax2.legend()
ax2.set_ylim(0, 1)
plt.tight_layout()
plt.show()


## 1.4 วัดผลบน test set — ครั้งแรกที่โมเดลเห็นข้อมูลเหล่านี้


In [ ]:
from sklearn.metrics import confusion_matrix  # noqa: E402

@torch.no_grad()
def predict_all(net, X, batch=512):
    net.eval()
    preds = []
    for i in range(0, len(X), batch):
        logits = net(X[i:i + batch].to(DEVICE))
        preds.append(logits.argmax(dim=-1).cpu())
    return torch.cat(preds).numpy()

y_pred = predict_all(model, X_test)
correct = int((y_pred == y_test.numpy()).sum())
test_acc = correct / len(y_test)
print(f"test accuracy: {test_acc:.3f} ({correct}/{len(y_test)})")

SHORT = [
    "down", "go", "left", "no", "off",
    "on", "right", "stop", "up", "yes", "unk", "sil",
]
cm = confusion_matrix(y_test.numpy(), y_pred)
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(SHORT)))
ax.set_xticklabels(SHORT, rotation=45)
ax.set_yticks(range(len(SHORT)))
ax.set_yticklabels(SHORT)
ax.set_xlabel("โมเดลทำนาย")
ax.set_ylabel("คำตอบจริง")
fig.colorbar(im)
plt.tight_layout()
plt.show()

pairs = []
for i in range(len(SHORT)):
    for j in range(len(SHORT)):
        if i != j and cm[i, j] > 0:
            pairs.append((int(cm[i, j]), SHORT[i], SHORT[j]))
pairs.sort(reverse=True)
print("คู่ที่สับสนมากที่สุด 8 อันดับ (จริง -> ทำนาย):")
for n, a, b in pairs[:8]:
    print(f"  {a:>6} -> {b:<6} : {n} ครั้ง")


**อ่าน confusion matrix:** แถบทแยงสีเข้ม = ตอบถูก ช่องนอกแนวทแยง = ความผิดพลาด สังเกตว่า `_unknown_` (unk) มักถูกสับสนกับคำสั่งบางคู่มากที่สุด — จำไว้สำหรับแล็บ


In [ ]:
from IPython.display import Audio, display  # noqa: E402

rng_demo = random.Random(7)
demo_idx = rng_demo.sample(range(len(test_pairs)), 3)
for i in demo_idx:
    path, label = test_pairs[i]
    print(
        f"จริง: {label} | โมเดลทำนาย: "
        f"{LABELS[int(y_pred[i])]}"
    )
    display(Audio(load_wave(path), rate=SAMPLES))


**สรุปส่วนที่ 1:** เราเพิ่งพาโมเดลเสียงตัวหนึ่ง "เกิด" ตั้งแต่ไฟล์ .wav ดิบ ๆ — แปลงเป็น log-mel → เทรน CNN จากค่า random → วัดผลจริงด้วย test set แยกตามผู้พูด ตัวเลข accuracy ที่ได้ **อ่านได้จริง** เพราะมาจากข้อมูลที่โมเดลไม่เคยเห็น ไม่ใช่การท่องจำ

## ส่วนที่ 2 — Demucs: แยกเพลงเป็น 4 stems (demo)

จุดเปลี่ยนของส่วนนี้: เราไม่เทรนเองแล้ว แต่ใช้โมเดลสำเร็จรูป **Demucs (htdemucs)** ของ Meta — โมเดลที่เทรนมาแล้วหลายพันชั่วโมงเพลงเพื่อ "แยก" เพลงเป็น 4 แทร็ก: กลอง / เบส / เครื่องดนตรีอื่น ๆ / เสียงร้อง

**ลองกับเพลงของคุณเอง:** ลากไฟล์ .mp3/.wav ใส่หน้า notebook ของKaggle (Drag & Drop ลงโฟลเดอร์ `/kaggle/working` หรือกด + Input → Upload) แล้วรันเซลล์ถัดไป — ถ้าไม่มีไฟล์ โค้ดจะสร้าง "เพลงตัวอย่างสังเคราะห์" ให้อัตโนมัติเพื่อให้ demo ยังรันจบ


In [ ]:
import glob  # noqa: E402

PATTERNS = [
    "/kaggle/working/*.mp3", "/kaggle/working/*.wav",
    "/kaggle/working/*.m4a", "/kaggle/working/*.flac",
    "/kaggle/input/**/*.mp3", "/kaggle/input/**/*.wav",
    "/kaggle/input/**/*.m4a", "/kaggle/input/**/*.flac",
]
found = []
for pattern in PATTERNS:
    found += glob.glob(pattern, recursive=True)
song_path = found[0] if found else None
if song_path:
    print("ใช้เพลงที่อัปโหลด:", song_path)
else:
    print("ไม่พบไฟล์เพลง — จะใช้เพลงตัวอย่างสังเคราะห์แทน")


In [ ]:
def make_toy_song(seconds=8, sr=44100):
    """เพลงตัวอย่างที่แยกชิ้นง่าย: กลอง+เบส+อาร์เปจโจ+โซโล่."""
    t = np.arange(int(sr * seconds)) / sr
    out = np.zeros((2, len(t)), dtype=np.float32)
    rng_song = np.random.default_rng(3)
    for beat in np.arange(0, seconds, 0.5):
        i0 = int(beat * sr)
        n = int(0.05 * sr)
        burst = rng_song.uniform(-1, 1, n)
        env = np.exp(-np.arange(n) / (0.01 * sr))
        out[:, i0:i0 + n] += 0.6 * burst * env
    for step, freq in enumerate([55, 62, 49, 55] * 2):
        i0 = int(step * sr)
        n = int(0.9 * sr)
        tone = np.sin(2 * np.pi * freq * np.arange(n) / sr)
        env = np.minimum(np.arange(n) / (0.05 * sr), 1.0)
        env *= np.exp(-np.arange(n) / (0.7 * sr))
        out[:, i0:i0 + n] += 0.5 * tone * env
    for step in range(int(seconds * 2)):
        freq = [440, 523, 659, 523][step % 4]
        i0 = int(step * 0.5 * sr)
        n = int(0.45 * sr)
        tone = np.sin(2 * np.pi * freq * np.arange(n) / sr)
        env = np.exp(-np.arange(n) / (0.3 * sr))
        out[:, i0:i0 + n] += 0.25 * 0.3 * tone * env
    vib = 220 * (1 + 0.03 * np.sin(2 * np.pi * 5 * t))
    env = 0.4 * (0.5 + 0.5 * np.sin(2 * np.pi * 0.5 * t))
    solo = np.sin(2 * np.pi * np.cumsum(vib) / sr)
    out += 0.35 * env * solo
    peak = np.abs(out).max()
    return (0.9 * out / peak).astype(np.float32)

if song_path is None:
    song_path = "/kaggle/working/toy_song.wav"
    toy = make_toy_song()
    sf.write(song_path, toy.T, 44100)
    print("บันทึกเพลงตัวอย่างที่:", song_path)


In [ ]:
import librosa  # noqa: E402
import torchaudio.functional as TF  # noqa: E402
from demucs.apply import apply_model  # noqa: E402
from demucs.pretrained import get_model  # noqa: E402

demucs_model = get_model("htdemucs")
demucs_model.to(DEVICE).eval()
TARGET_SR = demucs_model.samplerate
print("โมเดล: htdemucs | sample rate:", TARGET_SR,
      "| stems:", demucs_model.sources)

audio, sr = librosa.load(song_path, sr=None, mono=False)
if audio.ndim == 1 or audio.shape[0] == 1:
    audio = (
        np.repeat(audio, 2, axis=0)
        if audio.ndim > 1 else np.stack([audio, audio])
    )
elif audio.shape[0] > 2:
    audio = audio[:2]
wav = torch.from_numpy(audio.astype(np.float32))
if sr != TARGET_SR:
    wav = TF.resample(wav, sr, TARGET_SR)
MAX_SECONDS = 60
wav = wav[:, :TARGET_SR * MAX_SECONDS]
secs = round(wav.shape[1] / TARGET_SR, 1)
print("โหลดเพลง:", song_path, "|", secs, "วินาที")


Demucs เทรนมากับเสียง 44.1 kHz สองช่อง — โค้ดด้านบนจึง resample + ทำสองช่องให้เอง จากนั้นเรียก `apply_model` ซึ่งจัดการหั่นเพลงเป็นท่อน 8 วินาทีเอง (เพลง 60 วินาที ≈ 1–2 นาทีบน T4)


In [ ]:
ref = wav.mean(0)
wav_norm = (wav - ref.mean()) / (ref.std() + 1e-8)
with torch.no_grad():
    stems = apply_model(
        demucs_model,
        wav_norm[None].to(DEVICE),
        device=DEVICE,
        split=True,
        overlap=0.25,
        progress=True,
    )[0]
stems = stems * ref.std() + ref.mean()
print("รูปร่างผลลัพธ์:", tuple(stems.shape),
      "(stems x channels x samples)")


In [ ]:
TH_NAMES = {
    "drums": "กลอง",
    "bass": "เบส",
    "other": "เครื่องดนตรีอื่น ๆ",
    "vocals": "เสียงร้อง",
}
print("=== เพลงต้นฉบับ (mix) ===")
display(Audio(wav.cpu().numpy(), rate=TARGET_SR))
for k, name in enumerate(demucs_model.sources):
    print(f"=== stem: {name} ({TH_NAMES[name]}) ===")
    display(Audio(stems[k].cpu().numpy(), rate=TARGET_SR))


**ฟังแล้วเทียบ:** แต่ละ stem ควรมีเฉพาะส่วนของตัวเอง กดเล่น ทีละแทร็กเทียบกับ mix เดิม — นี่คือเครื่องมือจริงที่ใช้ใน สตูดิโอและงานวิจัยเสียง (karaoke ตัดเสียงร้อง, ฟอเรนซิกเสียง, การทำ dataset เสียง) สังเกต artifact ที่เหลือปนกันบ้าง — แยกได้ไม่ 100% และวัดคุณภาพการแยกด้วย SDR ได้ (คล้ายที่เราวัด accuracy/CER/WER)


In [ ]:
fig, axes = plt.subplots(
    len(demucs_model.sources) + 1, 1,
    figsize=(10, 8), sharex=True,
)
t = np.arange(wav.shape[1]) / TARGET_SR
axes[0].plot(t, wav[0].numpy(), linewidth=0.2)
axes[0].set_title("mix")
for k, name in enumerate(demucs_model.sources):
    axes[k + 1].plot(
        t, stems[k][0].cpu().numpy(), linewidth=0.2,
    )
    axes[k + 1].set_title(name)
for ax in axes:
    ax.set_ylabel("amp")
axes[-1].set_xlabel("วินาที")
plt.tight_layout()
plt.show()


## ส่วนที่ 3 — ASR bake-off: Whisper vs Typhoon-ASR-0.6B บน FLEURS-Thai

โจทย์สุดท้าย: **ถอดเสียงพูดไทยเป็นข้อความ — โมเดลไหนแม่นกว่า และเรารู้ได้ยังไง?** เราวัดด้วย **WER** บน FLEURS-Thai (ชุดอ่านบทจาก Wikimedia ของ Google ครอบคลุม 102 ภาษา — license CC BY 4.0)

ผู้เข้าแข่งสองคน:

| | Whisper-small | Typhoon-ASR-0.6B |
|---|---|---|
| ผู้ทำ | OpenAI | SCB 10X (ไทย) |
| ข้อมูลเทรน | เสียง ~680k ชั่วโมง **หลายภาษา** (ไทยเป็นส่วนน้อยมาก) | เสียงพูดไทย ~10,800 ชั่วโมง (GigaSpeech2-th) |
| สถาปัตยกรรม | encoder-decoder | Qwen3 LLM + audio encoder |
| ขนาด | 244M | ~0.6B (0.80B รวม embedding) |
| license | Apache-2.0 | Apache-2.0 |

สมมติฐานของโมดูล: โมเดลที่ "เรียนจากข้อมูลภาษาเป้าหมายจริง" ควรชนะโมเดลจากข้อมูลหลากภาษา — มาพิสูจน์ด้วยตัวเลข


In [ ]:
FLEURS_BASE = (
    "https://huggingface.co/datasets/google/fleurs/resolve/main/"
    "data/th_th"
)
FLEURS_ROOT = Path("/kaggle/working/fleurs_th")
tsv_path = FLEURS_ROOT / "dev.tsv"

if not (FLEURS_ROOT / "dev").exists():
    audio_tar = "/kaggle/working/fleurs_dev.tar.gz"
    if not Path(audio_tar).exists():
        print("กำลังดาวน์โหลด FLEURS-Thai dev ~247 MB ...")
        urllib.request.urlretrieve(
            f"{FLEURS_BASE}/audio/dev.tar.gz", audio_tar,
        )
    print("กำลังแตกไฟล์ ...")
    with tarfile.open(audio_tar) as tar:
        tar.extractall(FLEURS_ROOT, filter="data")
    Path(audio_tar).unlink()
if not tsv_path.exists():
    urllib.request.urlretrieve(f"{FLEURS_BASE}/dev.tsv", tsv_path)
print("คลิปเสียง dev:", len(list((FLEURS_ROOT / "dev").glob("*.wav"))))


เราใช้ **dev (validation) split** สำหรับ demo นี้ (เก็บ test split ไว้วัดจริงตอนสรุปผลงาน) เลือก 12 คลิปที่ยาว 5–15 วินาที แบบสุ่ม seed คงที่ — ทั้งหมด ~2–3 นาทีของเสียง พอสำหรับ "อ่านผลเปรียบเทียบ" โดยใช้เวลาไม่นาน ข้อสังเกต: คะแนนจาก 12 คลิปไม่ใช่ benchmark ทางการ — เราสอนการ**ตั้งชุดทดสอบแบบ ควบคุมได้**


In [ ]:
def clip_seconds(rec):
    try:
        return int(rec["num_samples"]) / 16000
    except (ValueError, TypeError):
        return 0.0

records = []
for line in tsv_path.read_text().splitlines():
    parts = line.split("\t")
    if len(parts) < 6:
        continue
    records.append(
        {"file": parts[1], "text": parts[2],
         "num_samples": parts[5]}
    )

rng_asr = random.Random(SEED)
N_EVAL = 12
cands = [r for r in records if 5.0 < clip_seconds(r) <= 15.0]
subset = rng_asr.sample(cands, N_EVAL)

waves = []
for r in subset:
    wave, sr_fleurs = sf.read(
        str(FLEURS_ROOT / "dev" / r["file"]),
        dtype="float32",
    )
    waves.append(wave.astype(np.float32))
total = sum(clip_seconds(r) for r in subset)
print(f"เลือก {N_EVAL} คลิป | sample rate: {sr_fleurs} Hz")
print(f"เวลารวม: {total:.1f} วินาที")
print("ตัวอย่างบทอ้างอิง:", subset[0]["text"])


### ผู้เข้า 1 — Whisper-small (ผ่าน transformers pipeline)


In [ ]:
from transformers import pipeline  # noqa: E402

WHISPER_ID = "openai/whisper-small"
whisper_pipe = pipeline(
    "automatic-speech-recognition",
    model=WHISPER_ID,
    device=0 if DEVICE.type == "cuda" else -1,
)
print("โหลด", WHISPER_ID, "แล้ว (~244M พารามิเตอร์)")

def whisper_transcribe(wave):
    out = whisper_pipe(
        wave,
        generate_kwargs={"language": "th", "task":
                         "transcribe"},
    )
    return out["text"].strip()

print("ตัวอย่างคลิปแรก:")
print("  อ้างอิง:", subset[0]["text"])
print("  Whisper:", whisper_transcribe(waves[0]))


### ผู้เข้า 2 — Typhoon-ASR-0.6B (โมเดลไทยจาก SCB 10X)

โมเดลนี้โหลดผ่าน API ใหม่ของ transformers 5.x (`AutoModelForMultimodalLM`) — บน T4 เราใช้ fp16 เพราะ T4 ไม่มี bf16 แบบ native


In [ ]:
from transformers import (  # noqa: E402
    AutoModelForMultimodalLM,
    AutoProcessor,
)

TYPHOON_ID = "typhoon-ai/typhoon-asr-qwen-0.6b-ctx"
typhoon_processor = AutoProcessor.from_pretrained(TYPHOON_ID)
if DEVICE.type == "cuda":
    dtype = (
        torch.bfloat16
        if torch.cuda.is_bf16_supported()
        else torch.float16
    )
    typhoon_model = AutoModelForMultimodalLM.from_pretrained(
        TYPHOON_ID, dtype=dtype, device_map="cuda",
    ).eval()
else:
    typhoon_model = AutoModelForMultimodalLM.from_pretrained(
        TYPHOON_ID,
    ).eval()
print("โหลด", TYPHOON_ID, "แล้ว (~0.6B พารามิเตอร์)")

@torch.no_grad()
def typhoon_transcribe(wave):
    """Typhoon-ASR: system 'Thai' + user audio -> ข้อความ."""
    msgs = [
        {"role": "system", "content": [
            {"type": "text", "text": "Thai"},
        ]},
        {"role": "user", "content": [
            {"type": "audio", "audio": wave},
        ]},
    ]
    inputs = typhoon_processor.apply_chat_template(
        [msgs],
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(typhoon_model.device)
    out = typhoon_model.generate(**inputs, max_new_tokens=440)
    out = out[:, inputs["input_ids"].shape[1]:]
    text = typhoon_processor.decode(
        out, return_format="transcription_only",
    )
    return text[0].strip()

print("ตัวอย่างคลิปแรก:")
print("  อ้างอิง:", subset[0]["text"])
print("  Typhoon:", typhoon_transcribe(waves[0]))


รันทั้ง 12 คลิปผ่านทั้งสองโมเดล — ใช้เวลาราว **3–6 นาทีบน T4** และพิมพ์ผลเทียบทีละคลิป (อ้างอิง vs Whisper vs Typhoon)


In [ ]:
import time  # noqa: E402

hyps_whisper, hyps_typhoon = [], []
for i, r in enumerate(subset):
    t0 = time.time()
    hyp_w = whisper_transcribe(waves[i])
    t_w = time.time() - t0
    t0 = time.time()
    hyp_t = typhoon_transcribe(waves[i])
    t_t = time.time() - t0
    hyps_whisper.append(hyp_w)
    hyps_typhoon.append(hyp_t)
    print(f"[{i + 1}/{N_EVAL}] {r['file']}")
    print("  อ้างอิง :", r["text"])
    print(f"  Whisper : {hyp_w}  ({t_w:.1f}s)")
    print(f"  Typhoon : {hyp_t}  ({t_t:.1f}s)")


## 3.1 วัดผลด้วย WER — ตัวเลขที่เทียบกันได้

`WER = (แทนคำ + ใส่เกิน + ตกหล่น) / จำนวนคำอ้างอิง` — เราใช้ jiwer เหมือนที่ใช้วัด CER ใน M2 พร้อม normalize เล็กน้อย (ตัด เครื่องหมาย `|` ของ FLEURS + จัดช่องว่าง) ให้ทั้งสองฝั่งอยู่ในเงื่อนไขเดียวกัน


In [ ]:
import jiwer  # noqa: E402

def norm_thai(text):
    text = text.replace("|", " ")
    return " ".join(text.split())

refs = [norm_thai(r["text"]) for r in subset]
hyps_w = [norm_thai(h) for h in hyps_whisper]
hyps_t = [norm_thai(h) for h in hyps_typhoon]

wer_w = jiwer.wer(refs, hyps_w)
wer_t = jiwer.wer(refs, hyps_t)
cer_w = jiwer.cer(refs, hyps_w)
cer_t = jiwer.cer(refs, hyps_t)
print(f"Whisper-small    : WER {wer_w:.1%} | CER {cer_w:.1%}")
print(f"Typhoon-ASR-0.6B : WER {wer_t:.1%} | CER {cer_t:.1%}")

names = ["Whisper-small", "Typhoon-ASR-0.6B"]
fig, ax = plt.subplots(figsize=(6, 3))
bars = ax.bar(names, [wer_w, wer_t])
ax.bar_label(bars, labels=[f"{wer_w:.0%}", f"{wer_t:.0%}"])
ax.set_ylabel("WER (ต่ำ = แม่น)")
top = max(wer_w, wer_t)
ax.set_ylim(0, top * 1.25 + 0.05)
ax.set_title("WER บน FLEURS-Thai dev (12 คลิปสุ่ม)")
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd  # noqa: E402

rows = []
for i in range(N_EVAL):
    rows.append({
        "clip": subset[i]["file"][:12],
        "wer_whisper": round(jiwer.wer(refs[i], hyps_w[i]), 2),
        "wer_typhoon": round(jiwer.wer(refs[i], hyps_t[i]), 2),
    })
df = pd.DataFrame(rows)
print(df.to_string(index=False))
print()
print("คลิปที่ Typhoon ชนะ:",
      int((df["wer_typhoon"] < df["wer_whisper"]).sum()),
      "/", N_EVAL)


**อ่านผล:** ตัวเลขจริงจะขึ้นกับ 12 คลิปที่สุ่มได้ แต่รูปแบบเดียวกันเกิดซ้ำทุกรอบ: Whisper-small พลาดหนักกับภาษาไทย (มักตอบคำอังกฤษ/จีน หรือถอดเพี้ยนเยอะ) ขณะที่ Typhoon-ASR ที่เทรนจากเสียงไทย ~10,800 ชั่วโมงให้ WER ต่ำกว่าชัดเจน — บทเรียนเดียวกับ M3: **ข้อมูลตรงโจทย์ > ขนาดโมเดล**

อีกจุดที่ควรสังเกต: WER นับ "คำ" ด้วยช่องว่าง ซึ่งภาษาไทยเขียนติดกัน — จึงควรดู **CER** คู่ไปด้วยเสมอ (เหมือนที่เราเตือนไว้ใน M2 เรื่อง OCR)


## สรุปโมดูล M5

| ตัวชี้วัด | โจทย์ | ผลที่เราอ่านได้ |
|---|---|---|
| accuracy + confusion matrix | KWS 12 class จากศูนย์ | ~0.90+ บน test แยกตามผู้พูด — และเห็นว่า `_unknown_` ยากสุด |
| หูของเราเอง | Demucs แยก stems | แยกได้จริงแต่ไม่สมบูรณ์ — artifact บอกขอบเขตของโมเดล |
| WER / CER | ASR ไทย 2 โมเดล | โมเดลที่เทรนด้วยข้อมูลไทยชนะ ชัดเจน |

ทั้งสามส่วนใช้วงจรเดียวกันทั้งคอร์ส: **ข้อมูล → โมเดล → ตัวชี้วัดที่อ่านได้** — และ "การวัด" คือสิ่งที่ทำให้เราเชื่อโมเดลได้อย่างมีเหตุผล (สไตล์เดียวกับงานเมโทรโลยี)

**อ้างอิงฉบับเต็ม**

- Speech Commands v2 — Pete Warden, Google — CC BY 4.0 — [ไฟล์ต้นทาง](http://download.tensorflow.org/data/speech_commands_v0.02.tar.gz) · [paper: arXiv:1804.03209](https://arxiv.org/abs/1804.03209)
- FLEURS — Conneau et al. 2022 — CC BY 4.0 — [HF: google/fleurs](https://huggingface.co/datasets/google/fleurs) · [paper: arXiv:2205.12447](https://arxiv.org/abs/2205.12447)
- Demucs / htdemucs — Meta AI — MIT — [GitHub](https://github.com/facebookresearch/demucs)
- Whisper-small — OpenAI — Apache-2.0 — [HF model card](https://huggingface.co/openai/whisper-small)
- Typhoon-ASR-0.6B (`typhoon-ai/typhoon-asr-qwen-0.6b-ctx`, LoRA-merged ของ Qwen3-ASR-0.6B-hf) — SCB 10X — Apache-2.0 — [HF model card](https://huggingface.co/typhoon-ai/typhoon-asr-qwen-0.6b-ctx)
- libraries: transformers (Apache-2.0), demucs (MIT), librosa (ISC), soundfile (BSD-3), jiwer (Apache-2.0), torch/torchaudio (BSD-3)
